In [ ]:
df[df['tweet_text'].str.contains('')]

# Método de Trabalho

Bibliotecas utilizadas no desenvolvimento do código

In [ ]:
import pandas as pd

from langdetect import detect
from collections import Counter

from urlextract import URLExtract
import re

from nltk.corpus import words
import nltk
nltk.download('words')

import emoji
from emot.emo_unicode import EMOTICONS_EMO

import unicodedata

import contractions


from spellchecker import SpellChecker
from textblob import TextBlob

from sklearn.model_selection import train_test_split

from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')

## Coleta dos Dados

In [ ]:
df = pd.read_csv("datasets/cyberbullying_tweets.csv")

df

## Pré-processamento dos Dados

### Amostragem representativa

### Limpeza

#### URLs

In [ ]:
# Verificar ocorrências de URLs

extractor = URLExtract()

def inspecionar_urls(texto):
    urls = []
    texto = str(texto)

    urls = extractor.find_urls(texto)

    if urls:
        print(urls)

df['tweet_text'].apply(inspecionar_urls)

In [ ]:
# Remover URLs

extractor = URLExtract()

def remover_urls(texto):
    texto = str(texto)

    urls = extractor.find_urls(texto)

    for url in urls:
        texto = texto.replace(url, "")

    return texto

df['tweet_text'] = df['tweet_text'].apply(remover_urls)

#### E-mails

In [ ]:
# Verificar ocorrências de e-mails

pattern = re.compile(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}')

def inspecionar_emails(texto):
    texto = str(texto)
    emails = pattern.findall(texto)

    if emails:
        print(emails)

df['tweet_text'].apply(inspecionar_emails)

In [ ]:
# Remover e-mails

pattern = re.compile(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}')

def remover_emails(texto):
    texto = str(texto)
    texto_limpo = pattern.sub("", texto)

    return texto_limpo

df['tweet_text'] = df['tweet_text'].apply(remover_emails)

#### Menção a user

In [ ]:
# Verificar ocorrências de menções a users

pattern = re.compile(r'(?:^| )(@[A-Za-z0-9_]+)')

def inspecionar_users(texto):
    texto = str(texto)
    users = pattern.findall(texto)

    if users:
        print(users, texto)

df['tweet_text'].apply(inspecionar_users)

In [ ]:
# Remover menções a users

pattern = re.compile(r'(?:^| )(@[A-Za-z0-9_@]+)')

def remover_users(texto):
    texto = str(texto)
    texto_limpo = pattern.sub("", texto)

    return texto_limpo

df['tweet_text'] = df['tweet_text'].apply(remover_users)

#### Espaços entre caracteres únicos consecutivos

In [ ]:
# Verificar ocorrências de espaços entre caracteres únicos consecutivos

pattern = re.compile(r'(?:\b\w\s){3,}\w\b')

def inspecionar_espacos(texto):
    texto = str(texto)
    espaco = pattern.findall(texto)

    if espaco:
        print(texto)

df['tweet_text'].apply(inspecionar_espacos)

In [ ]:
# Remover espaços entre caracteres únicos consecutivos

pattern = re.compile(r'(?:\b\w\s){3,}\w\b')

def remover_espacos(texto):
    texto = str(texto)
    texto_limpo = pattern.sub(lambda x: x.group().replace(" ", ""), texto)

    return texto_limpo

df['tweet_text'] = df['tweet_text'].apply(remover_espacos)

#### Múltiplos espaços

In [ ]:
# Remover múltiplos espaços

def remover_multiplos_espacos(texto):
    return " ".join(texto.split())

df['tweet_text'] = df['tweet_text'].apply(remover_multiplos_espacos)

#### Filtrar idiomas

In [ ]:
# Inspecionar idiomas presentes no df original

def detectar_idioma(texto):
    try:
        return detect(str(texto))
    except:
        return 'desconhecido'

idiomas = df['tweet_text'].apply(detectar_idioma)
contagem = Counter(idiomas)

for idioma, quantidade in contagem.most_common():
    print(f"{idioma}: {quantidade} tweets ({quantidade/len(df)*100:.1f}%)")

In [ ]:
# Criar novo df apenas da língua inglesa

def detectar_ingles(texto):
    try:
        return detect(str(texto)) == 'en'
    except:
        return False

df_ingles = df[df['tweet_text'].apply(detectar_ingles)]

In [ ]:
# Inspecionar idiomas presentes no df_ingles

def detectar_idioma(texto):
    try:
        return detect(str(texto))
    except:
        return 'desconhecido'

idiomas = df_ingles['tweet_text'].apply(detectar_idioma)
contagem = Counter(idiomas)

for idioma, quantidade in contagem.most_common():
    print(f"{idioma}: {quantidade} tweets ({quantidade/len(df)*100:.1f}%)")

### Amostragem

In [ ]:
# Amostragem estratificada - 10% de cada grupo

df_amostra, _ = train_test_split(
    df_ingles,
    test_size = 0.9,
    stratify = df_ingles['cyberbullying_type'],
    random_state = 42
)

print(f'Total: {len(df_amostra)}')
print(df_amostra['cyberbullying_type'].value_counts())

### Normalização

#### Normalização de caracteres alongados

In [ ]:
# Verificar ocorrências de caracteres alongados

pattern = re.compile(r'(\w)\1{2,}')

def inspecionar_caracteres_alongados(texto):
    texto = str(texto)
    espaco = pattern.findall(texto)

    if espaco:
        print(texto)

df_amostra['tweet_text'].apply(inspecionar_caracteres_alongados)

In [ ]:
# Remover caracteres alongados

palavras_validas = set(words.words())
pattern = re.compile(r'(\w)\1{1,}')

def remover_caracteres_alongados(texto):
    texto = str(texto)
    tokens = texto.split()
    resultado = []

    for token in tokens:
        if token.lower() not in palavras_validas:
            token = pattern.sub(r'\1', token)
        resultado.append(token)

    return " ".join(resultado)

df_amostra['tweet_text'] = df_amostra['tweet_text'].apply(remover_caracteres_alongados)

#### Transformação de emojis e emoticons em texto

In [ ]:
# Verificar ocorrências de emoticons

def inspecionar_emoticons(texto):
    texto = str(texto)
    encontrados = [emoticon for emoticon in EMOTICONS_EMO if emoticon in texto]
    
    if encontrados:
        print(texto)

df_amostra['tweet_text'].apply(inspecionar_emoticons)

In [ ]:
# Converter emoticons em texto

def converter_emoticons(texto):
    texto = str(texto)

    for emoticon, significado in EMOTICONS_EMO.items():
        texto = texto.replace(emoticon, significado)
        
    return texto

df_amostra['tweet_text'] = df['tweet_text'].apply(converter_emoticons)

In [ ]:
# Verificar ocorrências de emojis

def inspecionar_emojis(texto):
    texto = str(texto)

    if emoji.emoji_count(texto) > 0:
        print(texto)

df_amostra['tweet_text'].apply(inspecionar_emojis)

In [ ]:
# Converter emojis em texto

def converter_emojis():
    df_amostra['tweet_text'] = df_amostra['tweet_text'].apply(lambda x: emoji.demojize(str(x)))

converter_emojis()

#### Caracteres acentuados normalizados para o alfabeto inglês e em minúsculo

In [ ]:
# Remover acentos

def remover_acentos(texto):
    texto = unicodedata.normalize("NFD", texto)
    texto = ''.join(c for c in texto if unicodedata.category(c) != 'Mn')
    
    return texto

df_amostra['tweet_text'] = df_amostra['tweet_text'].apply(remover_acentos)

In [ ]:
# Texto para minúsculo

def para_lowercase(texto):
    return str(texto).lower()

df_amostra['tweet_text'] = df_amostra['tweet_text'].apply(para_lowercase)

#### Conversão de gírias para palavras normais

In [ ]:
# Criar dicionário de gírias

arquivo = 'datasets/girias.csv'

def criar_girias(arquivo):
    dicionario_girias = {}

    df_girias = pd.read_csv(arquivo)
    dicionario_girias.update(dict(zip(df_girias['giria'].str.lower(), df_girias['significado'].str.lower())))

    return dicionario_girias

In [ ]:
# Verificar ocorrências de gírias

dicionario_girias = criar_girias('datasets/girias.csv')

def inspecionar_girias(texto):
    texto = str(texto)
    tokens = texto.split()

    for token in tokens:
         if token.lower() not in palavras_validas:
               if dicionario_girias.get(token.lower()):
                    print(token)

df_amostra['tweet_text'].apply(inspecionar_girias)

In [ ]:
# Converter gírias para palavras normais

palavras_validas = set(words.words())
dicionario_girias = criar_girias('datasets/girias.csv')

def converter_girias(texto):
    texto = str(texto)
    tokens = texto.split()
    resultado = []

    for token in tokens:
        if token.lower() not in palavras_validas:
            token = dicionario_girias.get(token.lower(), token)
            
        resultado.append(token)
    
    return ' '.join(resultado)

df_amostra['tweet_text'] = df_amostra['tweet_text'].apply(converter_girias)

#### Expansão de contrações

In [ ]:
# Verificar ocorrências de contrações

lista_contracoes_reais = [
    chave for chave, valor in contractions.contractions_dict.items() 
    if chave.lower() != valor.lower()
]

regex_string = r'\b(' + '|'.join([re.escape(chave) for chave in lista_contracoes_reais]) + r')\b'
pattern_contracoes = re.compile(regex_string, flags=re.IGNORECASE)

def inspecionar_contracoes(texto):
    texto = str(texto)
    contracoes_encontradas = pattern_contracoes.findall(texto)
    
    if contracoes_encontradas:
        print(texto)

df_amostra['tweet_text'].apply(inspecionar_contracoes)

In [ ]:
# Expandir contrações

def expandir_contracoes(texto):
    texto = str(texto)
    texto_expandido = contractions.fix(texto)
    
    return texto_expandido

df_amostra['tweet_text'] = df_amostra['tweet_text'].apply(expandir_contracoes)

#### Correção ortográfica

In [ ]:
# Verificar ocorrências de erros ortográficos

spell = SpellChecker(language='en')
palavra_pattern = re.compile(r'\b[a-zA-Z]+\b')

def inspecionar_ortografia(texto):
    texto = str(texto)
    todas_palavras = palavra_pattern.findall(texto.lower())
    erros_encontrados = spell.unknown(todas_palavras)
    
    if erros_encontrados:
        print(f"{list(erros_encontrados)} em: {texto}")

df_amostra['tweet_text'].apply(inspecionar_ortografia)

In [ ]:
# Corrigir erros ortográficos

def corrigir_ortografia(texto):
    texto = str(texto)
    texto_corrigido = str(TextBlob(texto).correct())
    
    return texto_corrigido

df_amostra['tweet_text'] = df_amostra['tweet_text'].apply(corrigir_ortografia)

In [ ]:
# BACKUP ATÉ AQUI

df_backup = df_amostra

### Remoção e Lematização

#### Remoção de números

In [ ]:
# Verificar ocorrências de números

pattern = re.compile(r'\d+')

def inspecionar_numeros(texto):
    texto = str(texto)
    numeros = pattern.findall(texto)

    if numeros:
        print(numeros, texto)

df_amostra['tweet_text'].apply(inspecionar_numeros)

In [ ]:
# Remover números

pattern = re.compile(r'\d+')

def remover_numeros(texto):
    texto = str(texto)
    texto_limpo = pattern.sub("", texto)

    return texto_limpo

df_amostra['tweet_text'] = df_amostra['tweet_text'].apply(remover_users)

#### Remoção de pontuações

In [ ]:
# Verificar ocorrências de pontuações

pattern = re.compile(r'[^\w\s]')

def inspecionar_pontuacoes(texto):
    texto = str(texto)
    pontuacoes = pattern.findall(texto)

    if pontuacoes:
        print(pontuacoes, texto)

df_amostra['tweet_text'].apply(inspecionar_pontuacoes)

In [ ]:
# Remover pontuações

pattern = re.compile(r'[^\w\s]')

def remover_pontuacoes(texto):
    texto = str(texto)
    texto_limpo = pattern.sub("", texto)

    return texto_limpo

df_amostra['tweet_text'] = df_amostra['tweet_text'].apply(remover_pontuacoes)

#### Remoção de múltiplos espaços

In [ ]:
# Remover múltiplos espaços

def remover_multiplos_espacos(texto):
    return " ".join(texto.split())

df_amostra['tweet_text'] = df_amostra['tweet_text'].apply(remover_multiplos_espacos)

#### Lematização

In [ ]:
lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

def lematizar(texto):
    texto = str(texto)
    tokens = nltk.word_tokenize(texto)
    tags = nltk.pos_tag(tokens)
    resultado = []
    
    for token, tag in tags:
        if tag in ('PRP', 'PRP$'):
            continue
        pos = get_wordnet_pos(tag)
        lema = lemmatizer.lemmatize(token, pos)

        if lema == 'be':
            continue
        
        resultado.append(lema)
    
    return ' '.join(resultado)

df_amostra['tweet_text'] = df_amostra['tweet_text'].apply(lematizar)